# **Mineral Prospectivity Project**
## 01 data ingestion 

goals:\
-load Idaho State Survey and USGS data\
-identify and filter to data relevant to gold mining and prospectivity\
-export gold data subset for further analysis

### Part 1. import packages and identify local directory
import packages needed for data loading and analysis

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import glob
import os

data_path = "/Users/adbyerly/prospectivity_model/data/"

### Part 2. load raw data

a. load raw data from Idaho Geologic Survey mines database\
-load data\
-inspect format and identify data relevant to gold\
-filter to gold data\
-export layer file for future analysis

In [ ]:
#shapefile import
mines_gdf = gpd.read_file(data_path + "raw/MinesPubDD-1_2023/MinesPubDD-1_Shapefile_2023/Mines_DD-1_2023.shp")
mines_gdf.head()

In [ ]:
mines_gdf.shape

In [ ]:
mines_gdf.columns

In [ ]:
mines_gdf[mines_gdf["CommodityL"].str.contains("gold", case=False, na=False)]["CommodityL"].value_counts()

In [ ]:
#create a subset of this dataframe that only includes data relevant to gold
gold_gdf = mines_gdf[mines_gdf["CommodityL"].str.contains("gold", case=False, na=False)]
gold_gdf.head()

In [ ]:
# save gold layer
gold_gdf.to_file(data_path + "processed/idaho_gold_mines.geojson", driver="GeoJSON")

b. load raw geochemical data from the "USGS National Geochemical Database (NGDB)"\
-data filtered to the state of Idaho before bulk download from USGS website (https://www.usgs.gov/centers/gggsc/science/national-geochemical-database)\
-load data


In [ ]:
# csv - load all geochemical files from the downloaded folder into a dictionary
files = glob.glob(data_path + "raw/ngdbrock-fUS16/*.txt")
ngdb_dfs={}

for file in files:
    name = os.path.basename(file).replace(".txt", "")
    df = pd.read_csv(file, low_memory=False)
    ngdb_dfs[name] = df

ngdb_dfs.keys()

In [ ]:
# rename keys to more useful names based on metadata here: https://mrdata.usgs.gov/metadata/ngdbrock.html
renaming = {"tblRockGeoData" : "Rock_Data", #rock sample data
            "xtbOtherChem" : "Other_chem", #atomic absorption spectrometry, colorimetry, ion selective electrode (except Cl & F), fluorometry, or fire assay
            "xtbEsChem" : "ES", #electron spectroscopy
            "xtbUnknownChem" : "Unknown_chem", #analyses derived from undocumented methods
            "xtbIcpmsChem" : "ICPMS", #inductively coupled plasma mass spectrometry
            "xtbIcpaesChem" : "ICPAES", #inductively coupled plasma-atomic emission spectrometry
            "xtbMajorChem" : "Major_element", #major element oxide analyses plus forms-of-carbon, forms-of-sulfur, chlorine, and fluorine data.
            "xtbXrfChem" : "XRF", #x-ray fluorescence
            "xtbNaaChem" : "NAA" #instrumental neutron activation analysis or delayed neutron activation analysis
           }

ngdb = {renaming.get(k, k): v for k, v in ngdb_dfs.items()} 
ngdb.keys()

In [ ]:
# examine dataframes
for name, df in ngdb.items():
    print(name)
    print(df.head())
    print(df.columns)
    print(df.shape)
    print("-" * 25)